# METHOD 2 — Forests & Better Representations
### What if one tree is unstable? What if the pixels are the problem?

This notebook is fully implemented. Run, inspect, and copy reusable code to the Final Challenge.

**Presenter:** 8 minutes. Explain forest voting and compare raw pixels with HOG features.

In [ ]:
# Shared setup: works from local Jupyter or a fresh Google Colab runtime.
import subprocess, sys
from pathlib import Path
here = Path.cwd().resolve()
candidates = [here, here.parent, here / "doodle-ml-workshop", here.parent / "doodle-ml-workshop"]
ROOT = next((p for p in candidates if (p / "src" / "data.py").exists()), None)
if ROOT is None:
    checkout = Path("/content/Computer-Vision-Workshop")
    if not checkout.exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", "doodle-ai-workshop",
            "https://github.com/jaekop/Computer-Vision-Workshop.git", str(checkout)
        ], check=True)
    ROOT = checkout / "doodle-ml-workshop"
sys.path.insert(0, str(ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])
print("Workshop ready at", ROOT)

In [ ]:
from src.data import load_data
from src.evaluation import show_examples, validation_accuracy, show_mistakes
from src.models import fit_model
import matplotlib.pyplot as plt
import numpy as np

data = load_data(cache_dir=ROOT / "data")
print("Categories:", ", ".join(data.class_names))
print("Training:", data.X_train.shape, "Validation:", data.X_val.shape)
print("Test is held back until the Final Challenge.")
show_examples(data.X_train, data.y_train, data.class_names, count=12)
plt.show()

## 1. Many trees can vote
A Random Forest fits many randomized trees using bootstrapped samples and randomized feature choices. Voting can reduce variance, but it is not guaranteed to outperform every single tree.

In [ ]:
tree_model = fit_model(data.X_train, data.y_train, "tree", max_depth=12)
forest_model = fit_model(data.X_train, data.y_train, "forest", n_estimators=80)
print("Tree validation accuracy:", round(validation_accuracy(tree_model, data.X_val, data.y_val), 3))
print("Forest validation accuracy:", round(validation_accuracy(forest_model, data.X_val, data.y_val), 3))
example = data.X_val[2:3]
votes = [int(t.predict(forest_model.features(example))[0]) for t in forest_model.estimator.estimators_[:12]]
print("First 12 tree votes:", [data.class_names[v] for v in votes])
print("Forest final prediction:", data.class_names[int(forest_model.predict(example)[0])])

## 2. Raw pixels have a weakness
The same object shifted a few pixels can look very different to raw-pixel distance. Centering is a simple form of preprocessing, but not always beneficial.

In [ ]:
from src.features import center_ink, build_features
i = 10
original = data.X_val[i:i+1]
centered = center_ink(original)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(original.reshape(28, 28), cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(centered.reshape(28, 28), cmap="gray")
axes[1].set_title("Centered ink")
for ax in axes: ax.axis("off")
plt.show()
print("Raw-pixel difference after centering:",
      round(float(np.linalg.norm(build_features(original) - build_features(centered))), 3))

## 3. HOG: describe edges, not just pixels
Histogram of Oriented Gradients (HOG) summarizes edge directions in local regions. It is a classical engineered feature representation, not a deep network.

In [ ]:
from skimage.feature import hog
image = data.X_val[0].reshape(28, 28).astype(float) / 255
descriptor, visualization = hog(
    image, orientations=9, pixels_per_cell=(4, 4),
    cells_per_block=(2, 2), block_norm="L2-Hys", visualize=True
)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("28 × 28 drawing")
axes[1].imshow(visualization, cmap="magma")
axes[1].set_title(f"HOG edge summary ({len(descriptor)} features)")
for ax in axes: ax.axis("off")
plt.show()

## 4. Compare representations fairly
Use the **same train/validation split** and the same Random Forest family. Only change the representation or centering. HOG is not guaranteed to win.

In [ ]:
variants = [
    ("Raw pixels", "pixels", False),
    ("Centered pixels", "pixels", True),
    ("HOG", "hog", False),
]
feature_scores = {}
for label, representation, center in variants:
    model = fit_model(data.X_train, data.y_train, "forest",
                      representation=representation, center=center,
                      n_estimators=60)
    score = validation_accuracy(model, data.X_val, data.y_val)
    feature_scores[label] = score
    print(f"{label:18s}: validation accuracy = {score:.3f}")
plt.bar(feature_scores.keys(), feature_scores.values(), color=["#9B7BFF","#6DE4F4","#FFAB70"])
plt.ylim(0, 1); plt.ylabel("Validation accuracy")
plt.xticks(rotation=10); plt.title("The representation changes the experiment")
plt.show()

## 5. Learn from mistakes
Try moving a drawing, rotating it, or drawing with an unusual style. What is the model actually sensitive to?

In [ ]:
show_mistakes(forest_model, data.X_val, data.y_val, count=12)
plt.show()

### Copy this into the Final Challenge
- Use `fit_model(..., "forest", n_estimators=80)`.
- Try `representation="hog"` or `center=True`.
- Change **one thing at a time** and inspect validation mistakes.